# 04 | RAG中的CSV文档加载（CSVLoader）

`Document loader` 可以先理解成“资料读取器”。

它负责把外部资料读进来，比如 CSV、PDF、网页、Markdown，然后统一包装成 LangChain 认识的 `Document` 对象。

它不负责回答问题，也不负责向量检索。它做的事情比较朴素：**先把资料搬进来，并整理成后续 RAG 流程能处理的格式。**

所以在 RAG 里，Document loader 通常就是第一步。

这篇先看 CSV，它很适合做入门例子，因为它在真实业务里太常见了：客户反馈、订单明细、工单列表、招聘岗位、资产清单，很多系统最后都能导出一份 CSV。朴素，甚至有点土，但非常能打。


## 一、先把位置摆清楚

一条典型 RAG 流程大概是这样：

```text
CSV / PDF / 网页 / 数据库
-> Document loader
-> Document
-> Text splitter
-> Embedding
-> Vector store
-> Retriever
-> LLM 回答
```

`CSVLoader` 只负责前面这一步：

```text
本地 CSV 文件 -> 一组 Document
```

官方文档里也说得很直接：CSV loader 会把 CSV 数据加载成 Document，默认是一行数据对应一个 Document。

后面能不能检索得好、回答得准，那是 splitter、embedding、retriever 和 prompt 的事情。Loader 只是先把资料搬进门，所以别让它背全家的锅。

## 二、场景：客服反馈 CSV

假设一个电商团队每天都会收到用户反馈，比如退款、发票、商品质量、会员续费这些问题。

运营同事把反馈导出成 CSV，想做一个简单的 RAG 问答：

```text
最近用户主要在抱怨什么？
哪些反馈和退款有关？
有没有高优先级的问题？
```

这时第一步不是马上让模型回答，而是先把 CSV 里的每一条反馈变成 `Document`。

这篇就从这个最小场景开始：先把本地 CSV 加载进来，再看每一行会变成什么。

## 三、准备一份本地 CSV

我们先在本地生成一份小 CSV。真实项目里，这一步通常换成客服系统、问卷系统、运营后台导出的文件。说白了，就是“把一堆用户反馈导出成表格”。

In [ ]:
from pathlib import Path

# 统一把示例数据放到 rag/data 目录，避免到处散落。
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

csv_path = data_dir / "customer_feedback.csv"

# 一份很小的客服反馈表。
# ticket_id 是反馈编号，用来定位具体是哪一条反馈。
# 每一行代表一条用户反馈，后面 CSVLoader 会把每行转成一个 Document。
csv_path.write_text(
    "ticket_id,product,channel,priority,feedback\n"
    "T-1001,蓝牙耳机,电商客服,high,用户说耳机还没发货，想立刻退款\n"
    "T-1002,智能手表,App反馈,medium,用户找不到睡眠数据导出入口\n"
    "T-1003,无线键盘,电话客服,low,用户咨询键盘是否支持 Mac 快捷键\n"
    "T-1004,蓝牙耳机,电商客服,high,用户收到商品后发现左耳没有声音\n"
    "T-1005,会员服务,App反馈,medium,用户不理解自动续费规则\n",
    encoding="utf-8",
)

print(csv_path)

## 四、用 `CSVLoader` 加载 CSV

标准用法很短：创建 `CSVLoader`，然后调用 `load()`。

需要注意安装包：

```bash
uv add langchain-community
```

`CSVLoader` 在 `langchain_community` 里，不在 `langchain_core` 里。

In [ ]:
from langchain_community.document_loaders.csv_loader import CSVLoader

# 创建 CSVLoader，指向本地 CSV 文件。
loader = CSVLoader(file_path=str(csv_path), encoding="utf-8")

# load() 会一次性把 CSV 读完。
# 默认情况下，一行 CSV 会变成一个 Document。
documents = loader.load()

print(len(documents))
print(type(documents[0]))

现在得到的 `documents` 是一个列表，里面每个元素都是 `Document`。

一个 `Document` 最重要的两个部分是：

- `page_content`：真正要给后续 RAG 使用的文本内容
- `metadata`：来源、行号等辅助信息

先看第一条。

In [ ]:
first_doc = documents[0]

print("page_content:")
print(first_doc.page_content)

print("\nmetadata:")
print(first_doc.metadata)

这就是 CSVLoader 做的转换：把一行结构化表格，变成一段适合后续处理的文本。

它不是在“理解”这条反馈。它只是把资料整理成 LangChain 后续组件能接住的样子。

## 五、`source_column` 的进阶作用

前面加载出来的第一条反馈里，有一个 `ticket_id: T-1001`。

现在看默认情况：`metadata["source"]` 通常是 CSV 文件路径。

这在技术上没错，但业务上不够细。比如用户问：

```text
哪些反馈和退款有关？
```

如果系统最后只告诉你：

```text
来源：customer_feedback.csv
```

这句话帮助不大。因为 CSV 里有很多行，你还是不知道具体是哪一条反馈支撑了这个回答。

但是如果像下面这样，`source_column="ticket_id"` 后，它的作用就是把每一行里的反馈编号放进 `metadata["source"]`。

这样后面 RAG 检索到某个 Document 时，系统能知道它来自 `T-1001` 这条反馈，而不是只知道它来自整个 CSV 文件。

In [ ]:
# 先看默认 loader 的 source：通常只是 CSV 文件路径。
print("默认 source:")
print(documents[0].metadata)

# source_column 会把指定列写入 metadata["source"]。
# 这里用 ticket_id，因为反馈编号比文件名更具体，后续更容易定位到某一条反馈。
loader_with_source = CSVLoader(
    file_path=str(csv_path),
    encoding="utf-8",
    source_column="ticket_id",
)

documents_with_source = loader_with_source.load()

print("\n使用 source_column 后:")
print(documents_with_source[0].page_content)
print(documents_with_source[0].metadata)

这个例子里，`source_column="ticket_id"` 的效果更有业务意义。因为 `T-1001` 就像一个定位标签：

- 运营同事可以回到表格里找到这一行
- 客服同事可以继续查看这条反馈的完整上下文
- 系统后面也可以基于这个编号做跳转、复核、更新

## 六、大文件时用 `lazy_load()`

`load()` 会一次性读完整个 CSV。小文件没问题，但如果是几十万行的日志、订单、反馈记录，就不太优雅了。

这时候可以用 `lazy_load()`，一条一条地拿 Document。

In [ ]:
# lazy_load() 返回一个迭代器，不会一口气把所有 Document 都放进内存。
# 这里为了演示，只取前 2 条。
for index, document in enumerate(loader_with_source.lazy_load()):
    print(document.metadata)
    print(document.page_content)
    print("-" * 40)

    if index >= 1:
        break

`lazy_load()` 很适合后面接批处理，比如一批一批写入向量数据库。

当然，这篇先不往后走。本篇的重点只有一个：**先把 CSV 可靠地变成 Document。**

## 七、加载之后下一步是什么

现在我们拿到的是 `Document`，RAG 的第一步完成了。

后面通常还会继续做：

```text
Document
-> 切分 / 清洗
-> Embedding
-> 写入向量库
-> 检索相关反馈
-> 交给 LLM 组织回答
```